### Still not finished yet

In [5]:
from fastmcp import Client

from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery

from langsmith import traceable, get_current_run_tree

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage, ToolMessage, convert_to_openai_messages

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional
from IPython.display import Image, display
from operator import add
from openai import OpenAI

from utils.utils import format_ai_message

import openai

import random
import ast
import inspect
import instructor
import json

### List available tools in MCP servers running on http://localhost:8001/mcp and http://localhost:8002/mcp

In [6]:
mcp_client = Client("http://localhost:8001/mcp")

In [7]:
async with mcp_client:
    tools = await mcp_client.list_tools()

RuntimeError: Client failed to connect: All connection attempts failed

In [ ]:
tools

In [ ]:
print("==========NAME==========")
print(tools[0].name)
print("==========DESCRIPTION==========")
print(tools[0].description)
print("==========INPUT SCHEMA==========")
print(tools[0].inputSchema)

### Execute a tool on one of the running MCP Servers

In [ ]:
client = Client("http://localhost:8001/mcp")

async with client:
    
    result = await client.call_tool("get_formatted_items_context", {"query": "What earphones can I get?", "top_k": 5})

In [ ]:
result

In [ ]:
print(result.content[0].text)

### A function to extract tool definitions of all available tools in provided MCP Servers

In [ ]:
def parse_docstring_params(docstring: str) -> Dict[str, str]:
    """Extract parameter descriptions from docstring (handles both Args: and Parameters: formats),"""
    params = {}
    lines = docstring.split('n')
    in_params = False
    current_param = None

    for line in lines:
        stripped = line.strip()

        # Check for parameter section start
        if stripped in {'Args:', 'Arguments:', "Parameters:", 'Params:'}:
            in_params = True
            current_param = None
        elif stripped.startswith("Returns:") or stripped.startswith('Raises:'):
            in_params = False
        elif in_params:
            # Parse parameter Line (handles: "param: desc" and "- param: desc" formats)
            if ':' in stripped and (stripped[0].isalpha() or stripped.startswith(('-', '*'))):
                param_name = stripped.lstrip('- *').split(":")[0].strip()
                param_desc = ':'.join(stripped.lstrip('- *').Split(":")[1:]).strip()
                params[param_name] = param_desc
                current_param = param_name
            elif current_param and stripped:
                # Continuatian of previous parameter description
                params[current_param] += ' ' + stripped
                
    return params